# Step 4 — Analytical Sample Construction and Definitional Cleanup

## Purpose
Transform `city_311_final.csv` into a clean, analysis-ready flat file for Parts 2 and 3.

Three sub-steps:
- **4A** — Apply formal-311 definitional filter (automated screen + manual corrections)
- **4B** — Derive `adoption_year_clean` (integer or NaN, no strings) and `adoption_status`
- **4C** — Split into main analytical sample and review set

## Inputs
- `data_clean/city_311_final.csv` (314 rows, from step3_verify_and_merge.ipynb)

## Outputs
- `data_clean/city_311_analysis.csv` — main analytical sample (~300 rows)
- `data_clean/city_311_review.csv`   — excluded conflict/unclear cities (14 rows)

In [1]:
import re
import pandas as pd

CLEAN = '../data_clean/'

df = pd.read_csv(CLEAN + 'city_311_final.csv', dtype={'GEOID': str})
print(f'Loaded city_311_final.csv: {len(df)} rows')
print('\nstatus_final distribution:')
print(df['status_final'].value_counts().to_string())
print(f'\nCities with year_found: {df["year_found"].notna().sum()}')
print(f'Year range: {df["year_found"].min():.0f} – {df["year_found"].max():.0f}')

Loaded city_311_final.csv: 314 rows

status_final distribution:
status_final
adopted_year_unknown     138
adopted_year_verified    108
not_adopted               54
conflict_fp               13
not_adopted_unclear        1

Cities with year_found: 121
Year range: 1996 – 2025


## 4A — Formal-311 Definitional Filter

**Standard:** A city is coded `adopted` only if it operates a dedicated 311 non-emergency
phone number or platform **explicitly branded as "311"** by the city government.

**Automated screen:** For every city where `year_found` is not null, check whether the
string `"311"` appears in `evidence_note`. If it does not, the year is cleared and the
city is downgraded to `adopted_unknown_year`.

This catches cases where the search agent found a branded citizen-service app
(e.g., "Fix It Plano", "MyBellevue", "Engage Toledo") that is not formally designated
as a 311 system.

In [2]:
def has_311(note):
    return bool(re.search(r'311', str(note), re.IGNORECASE))

df['_note_has_311'] = df['evidence_note'].apply(has_311)

has_year   = df['year_found'].notna()
fail_auto  = df[has_year & ~df['_note_has_311']]

print(f'Cities with year_found          : {has_year.sum()}')
print(f'  Pass 311-string check          : {(has_year & df["_note_has_311"]).sum()}')
print(f'  Fail 311-string check          : {len(fail_auto)}')
print()
print('Cities failing automated screen:')
for _, r in fail_auto[['place_name', 'state_abbr', 'status_final', 'year_found', 'evidence_note']].iterrows():
    print(f'  {r.place_name}, {r.state_abbr} | year={r.year_found:.0f} | {r.status_final}')
    print(f'    {r.evidence_note}')

Cities with year_found          : 121
  Pass 311-string check          : 109
  Fail 311-string check          : 12

Cities failing automated screen:
  Plano city, TX | year=2011 | conflict_fp
    "Fix It Plano" citizen service app launched in July 2011, per NBC DFW and GovTech reporting.
  Toledo city, OH | year=2015 | adopted_year_verified
    "Engage Toledo" launched in October 2015 as Toledo's 24/7 citizen service request platform.
  Fort Collins city, CO | year=2011 | conflict_fp
    "Access Fort Collins" citizen service system was created in 2011, per Rocky Mountain Collegian reporting.
  Bellevue city, WA | year=2013 | adopted_year_verified
    Bellevue launched the MyBellevue app and citizen request portal in 2013, handling requests for streetlights, potholes, and other city issues.
  Joliet city, IL | year=2021 | adopted_year_verified
    City of Joliet launched CitizenVUE, a citizen engagement service request app, on April 7, 2021, replacing prior service request methods.
  Da

### Manual corrections

Two cases require manual intervention beyond the automated screen:

1. **Bridgeport CT** (`conflict_fp`, year=2012): The 2012 evidence is for "BConnected",
   a citizen-reporting app that is explicitly NOT the 311 system — the evidence note itself
   states it was replaced in 2016 by "Bridgeport 311" (SeeClickFix). Correct the year to
   2016 and override `status_corrected` to `adopted_year_verified` (no longer a conflict
   with O'Byrne's 2012 cutoff).

2. **Raleigh NC** (`conflict_fp`, year=2011): SeeClickFix-based 311 platform (2011) later
   replaced by "Ask Raleigh" (2025). The string "311" appears in the evidence note, so
   the automated screen passes it. Year retained, but a definitional flag is added to
   document the uncertainty. Status stays `conflict_fp` → review set.

In [3]:
MANUAL_CORRECTIONS = {
    ('CT', 'Bridgeport city'): {
        'year_override':   2016,
        'status_override': 'adopted_year_verified',
        'definitional_note': (
            'BConnected app (2012) was a citizen-reporting app, not a formal 311 system. '
            '"Bridgeport 311" (SeeClickFix) launched 2016 under Mayor Ganim. '
            'Year corrected from 2012 to 2016; status_corrected overridden to adopted_year_verified.'
        ),
    },
    ('NC', 'Raleigh city'): {
        'year_override': 2011,   # retain, but flag
        'definitional_note': (
            'SeeClickFix-based 311 platform (2011) later rebranded as "Ask Raleigh" (2025). '
            'Year 2011 retained; 311 branding was present but system not maintained long-term. '
            'Remains in review set as conflict_fp.'
        ),
    },
}

df_work = df.copy()
df_work['year_corrected']    = df_work['year_found'].copy()
df_work['status_corrected']  = df_work['status_final'].copy()
df_work['definitional_note'] = pd.NA

# Apply manual corrections
for (state, name), corr in MANUAL_CORRECTIONS.items():
    mask = (df_work['state_abbr'] == state) & (df_work['place_name'] == name)
    assert mask.sum() == 1, f'Expected 1 match for ({state}, {name}), got {mask.sum()}'
    df_work.loc[mask, 'definitional_note'] = corr['definitional_note']
    if 'year_override' in corr:
        df_work.loc[mask, 'year_corrected'] = corr['year_override']
    if 'status_override' in corr:
        df_work.loc[mask, 'status_corrected'] = corr['status_override']
    yr  = corr.get('year_override', 'unchanged')
    st  = corr.get('status_override', 'unchanged')
    print(f'Manual correction: {name}, {state}  →  year={yr}, status={st}')

# Apply automated screen: year not null, 311 not in note, no manual correction yet
auto_fail_mask = (
    df_work['year_corrected'].notna()
    & ~df_work['_note_has_311']
    & df_work['definitional_note'].isna()
)
auto_note = (
    'Automated screen: evidence note lacks explicit "311" designation; '
    'system appears to be a branded citizen-service app without formal 311 branding. '
    'Year cleared; status downgraded to adopted_unknown_year.'
)
df_work.loc[auto_fail_mask, 'definitional_note'] = auto_note
df_work.loc[auto_fail_mask, 'year_corrected']    = float('nan')

print(f'\nAutomated screen downgraded {auto_fail_mask.sum()} cities')
print('Affected:')
for _, r in df_work[auto_fail_mask][['place_name', 'state_abbr', 'status_corrected']].iterrows():
    print(f'  {r.place_name}, {r.state_abbr} ({r.status_corrected})')

df_work = df_work.drop(columns=['_note_has_311'])

Manual correction: Bridgeport city, CT  →  year=2016, status=adopted_year_verified
Manual correction: Raleigh city, NC  →  year=2011, status=unchanged

Automated screen downgraded 12 cities
Affected:
  Plano city, TX (conflict_fp)
  Toledo city, OH (adopted_year_verified)
  Fort Collins city, CO (conflict_fp)
  Bellevue city, WA (adopted_year_verified)
  Joliet city, IL (adopted_year_verified)
  Dayton city, OH (adopted_year_verified)
  Victorville city, CA (adopted_year_verified)
  Midland city, TX (adopted_year_verified)
  Ann Arbor city, MI (adopted_year_verified)
  Clearwater city, FL (adopted_year_verified)
  West Jordan city, UT (adopted_year_verified)
  Manchester city, NH (adopted_year_verified)


## 4B — Derive Clean Analytical Columns

**`adoption_year_clean`** — integer year or `NaN`, no strings ever.
Populated only for cities where `adoption_status == 'adopted'`
(formal 311 confirmed, year reliable). All other rows are `NaN`.

**`adoption_status`** — three-value categorical for the main sample:
- `adopted` — formal 311, year known and reliable
- `adopted_unknown_year` — formal 311 confirmed, year not known
- `not_adopted` — no 311 system
- `NaN` — conflict/unclear → excluded to review set

In [4]:
def get_adoption_status(row):
    sc = row['status_corrected']
    yc = row['year_corrected']
    if sc in ('conflict_fp', 'not_adopted_unclear'):
        return pd.NA
    if sc in ('adopted_year_verified', 'adopted_year_unknown'):
        return 'adopted' if pd.notna(yc) else 'adopted_unknown_year'
    if sc == 'not_adopted':
        return 'not_adopted'
    return pd.NA

df_work['adoption_status'] = df_work.apply(get_adoption_status, axis=1)

# adoption_year_clean: integer only for formally adopted cities with known year
adopted_with_year = (df_work['adoption_status'] == 'adopted') & df_work['year_corrected'].notna()
df_work['adoption_year_clean'] = pd.NA
df_work.loc[adopted_with_year, 'adoption_year_clean'] = (
    df_work.loc[adopted_with_year, 'year_corrected'].astype(int)
)
df_work['adoption_year_clean'] = df_work['adoption_year_clean'].astype('Int64')

print('adoption_status distribution (incl. review set):')
print(df_work['adoption_status'].value_counts(dropna=False).to_string())
print()
known = df_work[df_work['adoption_year_clean'].notna()]
print(f'Cities with adoption_year_clean  : {len(known)}')
print(f'Year range                       : {int(known["adoption_year_clean"].min())} – {int(known["adoption_year_clean"].max())}')
print()
print('adoption_year_clean distribution:')
print(df_work['adoption_year_clean'].value_counts().sort_index().to_string())

adoption_status distribution (incl. review set):
adoption_status
adopted_unknown_year    148
adopted                  99
not_adopted              54
<NA>                     13

Cities with adoption_year_clean  : 99
Year range                       : 1996 – 2025

adoption_year_clean distribution:
adoption_year_clean
1996    1
1999    4
2000    3
2002    2
2003    4
2004    4
2005    4
2006    3
2007    6
2008    5
2009    3
2011    1
2012    3
2013    6
2014    3
2015    2
2016    3
2017    6
2018    6
2019    3
2020    5
2021    3
2022    7
2023    1
2024    3
2025    8


## 4C — Split Main Sample and Review Set

**Review set criteria:** `adoption_status` is `NaN`, i.e. `status_corrected ∈ {conflict_fp, not_adopted_unclear}`.

These cities are excluded from the main analytical sample because their adoption status
is genuinely ambiguous. Including them as either treated or control would introduce
measurement error in both groups. They are preserved separately for sensitivity checks.

The one exception is **Bridgeport CT**, which was a `conflict_fp` in Step 3 but was
manually resolved in 4A (year corrected to 2016, status_corrected = `adopted_year_verified`)
and is now in the main sample.

In [5]:
main_mask   = df_work['adoption_status'].notna()
review_mask = ~main_mask

main_sample = df_work[main_mask].reset_index(drop=True)
review_set  = df_work[review_mask].reset_index(drop=True)

assert len(main_sample) + len(review_set) == len(df), 'Row count mismatch'

print(f'Main analytical sample : {len(main_sample)} rows')
print(f'Review set             : {len(review_set)} rows')
print(f'Total                  : {len(main_sample) + len(review_set)}')
print()
print('Main sample — adoption_status:')
print(main_sample['adoption_status'].value_counts().to_string())
print()
print('Review set cities (all conflict_fp except Hialeah = not_adopted_unclear):')
cols_show = ['place_name', 'state_abbr', 'pop_2020', 'status_final', 'year_found', 'source_type']
for _, r in review_set[cols_show].sort_values('pop_2020', ascending=False).iterrows():
    print(f'  {r.place_name}, {r.state_abbr} | pop={r.pop_2020:,} | {r.status_final} | year_found={r.year_found} | src={r.source_type}')

Main analytical sample : 301 rows
Review set             : 13 rows
Total                  : 314

Main sample — adoption_status:
adoption_status
adopted_unknown_year    148
adopted                  99
not_adopted              54

Review set cities (all conflict_fp except Hialeah = not_adopted_unclear):
  El Paso city, TX | pop=679,255 | conflict_fp | year_found=2011.0 | src=news
  Boston city, MA | pop=675,466 | conflict_fp | year_found=2009.0 | src=official
  Las Vegas city, NV | pop=646,794 | conflict_fp | year_found=2000.0 | src=news
  Milwaukee city, WI | pop=577,207 | conflict_fp | year_found=2012.0 | src=news
  Raleigh city, NC | pop=465,354 | conflict_fp | year_found=2011.0 | src=news
  Cleveland city, OH | pop=371,806 | conflict_fp | year_found=2009.0 | src=official
  Plano city, TX | pop=286,408 | conflict_fp | year_found=2011.0 | src=news
  Winston-Salem city, NC | pop=249,804 | conflict_fp | year_found=2007.0 | src=news
  Hialeah city, FL | pop=222,408 | not_adopted_unclear |

In [6]:
# Quick sanity checks before saving

# 1. adoption_year_clean has no strings
assert df_work['adoption_year_clean'].dtype == pd.Int64Dtype(), 'adoption_year_clean should be Int64'

# 2. adopted cities all have a year
adopted = main_sample[main_sample['adoption_status'] == 'adopted']
assert adopted['adoption_year_clean'].notna().all(), 'Some adopted cities missing year_clean'

# 3. non-adopted cities have no year
not_adopted = main_sample[main_sample['adoption_status'] == 'not_adopted']
assert not_adopted['adoption_year_clean'].isna().all(), 'not_adopted cities should have NaN year_clean'

# 4. adopted_unknown_year cities have no year
unk = main_sample[main_sample['adoption_status'] == 'adopted_unknown_year']
assert unk['adoption_year_clean'].isna().all(), 'adopted_unknown_year cities should have NaN year_clean'

# 5. Bridgeport is in main sample with year 2016
bpt = main_sample[(main_sample['state_abbr'] == 'CT') & (main_sample['place_name'] == 'Bridgeport city')]
assert len(bpt) == 1, 'Bridgeport should be in main sample'
assert int(bpt['adoption_year_clean'].iloc[0]) == 2016, f'Bridgeport year should be 2016, got {bpt["adoption_year_clean"].iloc[0]}'

# 6. Review set has no adoption_year_clean values
assert review_set['adoption_year_clean'].isna().all(), 'Review set should have no adoption_year_clean'

print('All sanity checks passed.')

All sanity checks passed.


In [7]:
OUT_COLS = [
    'GEOID', 'place_name', 'state_abbr', 'pop_2020',
    'adoption_year_clean', 'adoption_status',
    'year_found', 'year_corrected',
    'status_final', 'status_corrected',
    'source_url', 'source_type', 'evidence_note',
    'obyrne_label', 'adopt_by2012',
    'audit_note', 'definitional_note',
]

main_sample[OUT_COLS].to_csv(CLEAN + 'city_311_analysis.csv', index=False)
print(f'Saved {len(main_sample)} rows → {CLEAN}city_311_analysis.csv')

review_set[OUT_COLS].to_csv(CLEAN + 'city_311_review.csv', index=False)
print(f'Saved {len(review_set)} rows  → {CLEAN}city_311_review.csv')

print()
print('=== STEP 4 COMPLETE ===')
print(f'Main sample : {len(main_sample)} cities')
print(f'  adopted              : {(main_sample["adoption_status"]=="adopted").sum():>4}  (formal 311, year known)')
print(f'  adopted_unknown_year : {(main_sample["adoption_status"]=="adopted_unknown_year").sum():>4}  (formal 311, year unknown)')
print(f'  not_adopted          : {(main_sample["adoption_status"]=="not_adopted").sum():>4}  (no 311 system)')
print(f'Review set  : {len(review_set)} cities (conflict_fp + not_adopted_unclear)')
print()
print('Key column types in city_311_analysis.csv:')
for col in ['adoption_year_clean', 'adoption_status', 'year_found', 'year_corrected']:
    print(f'  {col:<25} {str(main_sample[col].dtype)}')

Saved 301 rows → ../data_clean/city_311_analysis.csv
Saved 13 rows  → ../data_clean/city_311_review.csv

=== STEP 4 COMPLETE ===
Main sample : 301 cities
  adopted              :   99  (formal 311, year known)
  adopted_unknown_year :  148  (formal 311, year unknown)
  not_adopted          :   54  (no 311 system)
Review set  : 13 cities (conflict_fp + not_adopted_unclear)

Key column types in city_311_analysis.csv:
  adoption_year_clean       Int64
  adoption_status           object
  year_found                float64
  year_corrected            float64
